In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [3]:
df = pd.read_csv("../data/processed/loan_data_engineered.csv")
print(df.shape)

(1345350, 50)


In [4]:
import lightgbm as lgb
print(lgb.__version__)

4.7.0


In [5]:
drop_cols = ['loan_status', 'default']

In [8]:
X = df.drop(columns=drop_cols)
y = df['default']

print(X.shape)
print(y.value_counts(normalize=True)*100)

(1345350, 48)
default
0    80.035009
1    19.964991
Name: proportion, dtype: float64


In [15]:
categorical_cols = X.select_dtypes(include='str').columns.tolist()
print(categorical_cols)

['term', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type']


In [16]:
print('earliest_cr_line' in X.columns)
print(X.dtypes)

False
loan_amnt                     float64
term                              str
installment                   float64
emp_length                        str
home_ownership                    str
annual_inc                    float64
verification_status               str
purpose                           str
addr_state                        str
dti                           float64
fico_range_low                float64
fico_range_high               float64
inq_last_6mths                float64
mths_since_last_delinq        float64
mths_since_last_record        float64
open_acc                      float64
pub_rec                       float64
revol_bal                     float64
revol_util                    float64
total_acc                     float64
initial_list_status               str
application_type                  str
acc_now_delinq                float64
tot_coll_amt                  float64
tot_cur_bal                   float64
avg_cur_bal                   float64
bc_ope

In [17]:
for col in categorical_cols:
    X[col] = X[col].astype('category')

print(X.dtypes)

loan_amnt                      float64
term                          category
installment                    float64
emp_length                    category
home_ownership                category
annual_inc                     float64
verification_status           category
purpose                       category
addr_state                    category
dti                            float64
fico_range_low                 float64
fico_range_high                float64
inq_last_6mths                 float64
mths_since_last_delinq         float64
mths_since_last_record         float64
open_acc                       float64
pub_rec                        float64
revol_bal                      float64
revol_util                     float64
total_acc                      float64
initial_list_status           category
application_type              category
acc_now_delinq                 float64
tot_coll_amt                   float64
tot_cur_bal                    float64
avg_cur_bal              

In [18]:
from sklearn.model_selection import train_test_split

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [20]:
print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(1076280, 47) (269070, 47)
default
0    0.80035
1    0.19965
Name: proportion, dtype: float64
default
0    0.800349
1    0.199651
Name: proportion, dtype: float64


In [21]:
model = lgb.LGBMClassifier(
    random_state = 42,
    n_estimators = 100
)

In [22]:
model.fit(X_train, y_train, categorical_feature = categorical_cols)

print("Training Complete")

[LightGBM] [Info] Number of positive: 214879, number of negative: 861401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.499793 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5648
[LightGBM] [Info] Number of data points in the train set: 1076280, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.199650 -> initscore=-1.388485
[LightGBM] [Info] Start training from score -1.388485
Training Complete


In [24]:
from sklearn.metrics import roc_auc_score

In [25]:
y_pred_proba = model.predict_proba(X_test)[:,1]

In [28]:
auc = roc_auc_score(y_test, y_pred_proba)
print(f"Test AUC: {auc:.4f}")

Test AUC: 0.7219


## Baseline Model
# LightGBM with default hyperparameters, no imbalance handling
# Test AUC: 0.7219

In [29]:
import joblib

joblib.dump(model, '../models/baseline_lightgbm.pkl')
print("Model saved")

Model saved
